# Lab 3 - Solved: Generating tweets using n-grams

In [1]:
import os
import re
import emoji
import nltk
import pandas as pd
import kagglehub

from nltk.tokenize import word_tokenize
from nltk.lm import MLE, Laplace
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends
from nltk.util import ngrams
from sklearn.model_selection import train_test_split

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

C:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## 1. Load the data

In [2]:
path = kagglehub.dataset_download("adizafar/large-random-tweets-from-pakistan")
df = pd.read_csv(os.path.join(path, "Random Tweets from Pakistan- Cleaned- Anonymous.csv"), encoding_errors="ignore", low_memory=False)
df = df.dropna(subset=['full_text'])
df.head()

,Unnamed: 0,created_at_tweet,full_text,retweet_count,favorite_count,reply_count,location
0,0,Wed Jul 03 07:20:15 +0000 2019,تیرا لیڈر میرا لیڈر نواز شریف نواز شریف❤️ کیا...,226,1097,127,"Punjab, Pakistan"
1,1,Fri Jul 02 02:50:53 +0000 2021,"Happy birthday to my brother n boss , May you ...",54,273,9,"Punjab, Pakistan"
2,2,Fri Jul 02 12:12:53 +0000 2021,❤️❤️,50,131,13,"Punjab, Pakistan"
3,3,Tue Oct 15 22:29:16 +0000 2019,`suspicious °jikook au jimin'in yaşadığı kasab...,522,2134,86,"İstanbul, Türkiye"
4,4,Wed May 13 04:34:31 +0000 2020,Speaking of @SpiderMan... 😂 https://t.co/uGJ...,74,535,11,Follow the ATP Tour ➡


## 2. Pre-processing

In [3]:
def clean_tweet(text):
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"\bRT\b", "", text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = emoji.replace_emoji(text, replace="")
    return text.lower()

tokens = []
for tweet in df['full_text'].astype(str):
    tweet = clean_tweet(tweet)
    if re.search(r"[^\x00-\x7F]", tweet):
        continue
    words = [w for w in word_tokenize(tweet) if re.match(r"^[a-z0-9]+$", w)]
    if len(words) >= 2:
        tokens.append(words)

print(len(tokens))
print(tokens[:3])

101572
[['happy', 'birthday', 'to', 'my', 'brother', 'n', 'boss', 'may', 'you', 'have', 'many', 'many', 'more', 'you', 'are', 'gem'], ['speaking', 'of'], ['alexa', 'skip', 'to', '2021']]


## 3. Build MLE model (bigram)

In [4]:
train_tweets, test_tweets = train_test_split(tokens, test_size=0.1, random_state=42)

train, vocab = padded_everygram_pipeline(2, train_tweets)

lm = MLE(2)
lm.fit(train, vocab)
print(len(lm.vocab))

60990


In [5]:
for i in range(10):
    words = lm.generate(20, text_seed=['<s>'], random_seed=i)
    tweet = []
    for w in words:
        if w == '</s>':
            break
        tweet.append(w)
    print(' '.join(tweet))

tirah valley gilgit baltistan lt cdr aliya ali jinnah institute swabi ma end this not coming true whichever road to
breaking pregnant women cricket is liberated territories pti and accounts r k peshey par hona k bkwass burgers pizza
with which are back to see the future of poor performance and islamabad e ni yh tum log hotay hain


follow me exams lyny hain


five deaths for cancellation


pakistan presented their us safe than 10 loyal to poora saal ap ki celebrity has money


that this is dominating hammering on lies to give us all the release keeping ke paper cancel hi please convey


how can only 20 minutes in a meeting


faislabad today


khan from a tad bit now to attract much more


## 4. Evaluate the model

In [6]:
def to_bigrams(tweets):
    return [bg for t in tweets for bg in ngrams(pad_both_ends(t, n=2), n=2)]

print("Perplexity (train):", lm.perplexity(to_bigrams(train_tweets[:1000])))
print("Perplexity (test):", lm.perplexity(to_bigrams(test_tweets[:1000])))

train, vocab = padded_everygram_pipeline(2, train_tweets)
laplace = Laplace(2)
laplace.fit(train, vocab)
print("Laplace perplexity (test):", laplace.perplexity(to_bigrams(test_tweets[:1000])))

Perplexity (train): 66.46100975406046
Perplexity (test): inf


Laplace perplexity (test): 6466.80160722135


MLE gives unseen bigrams probability 0, so its test perplexity is infinite; Laplace smoothing fixes this.

## 5. Probability of the bigram (pakistan is)

In [7]:
print(lm.score('is', ['pakistan']))

0.04485286073733727


## 6. Perplexity of the word (pakistan)

In [8]:
print(lm.perplexity([('pakistan',)]))

173.7866754184444
